# Face Analytics диплом — готовий ноутбук
Цей ноутбук тренує **окремо** 3 ViT-моделі:
- **Emotion-only** (7 класів) на `dilkushsingh/facial-emotion-dataset`
- **Age-only** (9 age bins) на `jangedoo/utkface-new`
- **Gender-only** (woman/man) на `jangedoo/utkface-new`

Також є блок для **YOLO** на `aklimarimi/8-facial-expressions-for-yolo` з видаленням класів **Contempt** та **Sleepy**.

⚠️ У emotion-датасеті є **биті/недокачані** JPEG (інколи 300–500 bytes). Тут вони автоматично фільтруються, а Dataset додатково стійкий (не падає на одиничних файлах).


In [ ]:
# === Install (Kaggle/Colab) ===
# У Kaggle більшість пакетів вже є, але цей блок безпечний.
!pip -q install kagglehub pyyaml ultralytics
!pip -q install transformers torch torchvision torchaudio
!pip -q install opencv-python-headless numpy pandas scikit-learn matplotlib tqdm


In [ ]:
# === Imports + env helpers ===
import os, re, json, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoImageProcessor, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, roc_auc_score, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

def ensure_out_root():
    # Kaggle writes to /kaggle/working; Colab writes to /content
    return "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"

OUT_ROOT = ensure_out_root()
print("OUT_ROOT =", OUT_ROOT)

IMG_EXT = (".jpg",".jpeg",".png",".bmp",".webp")


In [ ]:
# === Download datasets via kagglehub ===
import kagglehub

# Emotion dataset (has train_dir/test_dir)
emo_root = kagglehub.dataset_download("dilkushsingh/facial-emotion-dataset")

# Age + Gender (UTKFace)
utk_root = kagglehub.dataset_download("jangedoo/utkface-new")

# YOLO (bbox + emotion classes)
yolo_root = kagglehub.dataset_download("aklimarimi/8-facial-expressions-for-yolo")

print("emo_root :", emo_root)
print("utk_root :", utk_root)
print("yolo_root:", yolo_root)


In [ ]:
# === Emotion DF builder (7 classes) ===
EMO_CLASSES_7 = ["angry","disgust","fear","happy","neutral","sad","surprise"]
EMO_TO_ID = {c:i for i,c in enumerate(EMO_CLASSES_7)}

emo_train_dir = os.path.join(emo_root, "train_dir")
emo_test_dir  = os.path.join(emo_root, "test_dir")

assert os.path.isdir(emo_train_dir), f"Missing: {emo_train_dir}"
assert os.path.isdir(emo_test_dir),  f"Missing: {emo_test_dir}"

def collect_emotion_split(split_dir: str) -> pd.DataFrame:
    rows = []
    for cls in EMO_CLASSES_7:
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            print("Missing class dir:", cls_dir)
            continue
        cls_id = EMO_TO_ID[cls]
        for dp, _, files in os.walk(cls_dir):
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    rows.append({"image_path": os.path.join(dp, fn), "label": cls_id})
    return pd.DataFrame(rows)

emo_train_df = collect_emotion_split(emo_train_dir)
emo_test_df  = collect_emotion_split(emo_test_dir)

print("emo_train_df:", len(emo_train_df))
print("emo_test_df :", len(emo_test_df))
print("train counts:\n", emo_train_df["label"].value_counts().sort_index())
print("test counts:\n", emo_test_df["label"].value_counts().sort_index())


In [ ]:
# === Remove broken images (cv2 decode) — recommended ===
def filter_bad_decode(df: pd.DataFrame, desc="decode check"):
    good = []
    bad = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        p = row["image_path"]
        img = cv2.imread(p, cv2.IMREAD_COLOR)
        if img is None:
            bad.append(p)
        else:
            good.append(row)
    return pd.DataFrame(good).reset_index(drop=True), bad

emo_train_df, bad_tr = filter_bad_decode(emo_train_df, "clean emotion train")
emo_test_df,  bad_te = filter_bad_decode(emo_test_df,  "clean emotion test")

print("Removed broken train:", len(bad_tr))
print("Removed broken test :", len(bad_te))
print("After clean -> train:", len(emo_train_df), "test:", len(emo_test_df))

# Optional: save bad lists
bad_dir = os.path.join(OUT_ROOT, "runs", "bad_images")
os.makedirs(bad_dir, exist_ok=True)
with open(os.path.join(bad_dir, "emotion_bad_train.txt"), "w") as f:
    f.write("\n".join(bad_tr))
with open(os.path.join(bad_dir, "emotion_bad_test.txt"), "w") as f:
    f.write("\n".join(bad_te))
print("Bad lists saved to:", bad_dir)


In [ ]:
# === Single-task ViT: dataset/model/training + metrics ===
def read_rgb_safe(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

class SingleTaskImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, processor, label_col: str):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        # robust: avoid crash on a single bad file
        for _ in range(5):
            row = self.df.iloc[idx]
            img = read_rgb_safe(row["image_path"])
            if img is not None:
                x = self.processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
                y = int(row[self.label_col])
                return x, y
            idx = np.random.randint(0, len(self.df))

        dummy = np.zeros((224,224,3), dtype=np.uint8)
        x = self.processor(images=dummy, return_tensors="pt")["pixel_values"].squeeze(0)
        return x, 0

def collate_single(batch):
    xs = torch.stack([b[0] for b in batch], dim=0)
    ys = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return xs, ys

class ViTClassifier(nn.Module):
    def __init__(self, backbone: str, n_classes: int):
        super().__init__()
        self.vit = AutoModel.from_pretrained(backbone)
        h = self.vit.config.hidden_size
        self.head = nn.Linear(h, n_classes)

    def forward(self, pixel_values):
        out = self.vit(pixel_values=pixel_values)
        cls = out.last_hidden_state[:, 0]
        return self.head(cls)

def make_class_weights(labels: np.ndarray, n_classes: int):
    w = compute_class_weight("balanced", classes=np.arange(n_classes), y=labels)
    w = w / np.mean(w)
    return torch.tensor(w, dtype=torch.float32)

@torch.no_grad()
def predict_proba(model, loader, device):
    model.eval()
    all_y, all_p = [], []
    for x, y in tqdm(loader, desc="predict", leave=False):
        x = x.to(device)
        logits = model(x)
        p = torch.softmax(logits, dim=1).cpu().numpy()
        all_p.append(p)
        all_y.append(y.numpy())
    return np.concatenate(all_y), np.concatenate(all_p)

def save_curves_and_reports(y_true, y_proba, class_names, out_dir, prefix):
    os.makedirs(out_dir, exist_ok=True)
    y_pred = y_proba.argmax(1)

    rep = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
    with open(os.path.join(out_dir, f"{prefix}_classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(rep)

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8,6))
    disp.plot(ax=ax, xticks_rotation=45, values_format="d")
    ax.set_title(f"{prefix} Confusion Matrix")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{prefix}_confusion_matrix.png"), dpi=150)
    plt.close(fig)

    C = len(class_names)
    stats = {}

    if C == 2:
        y_score = y_proba[:, 1]
        fpr, tpr, _ = roc_curve(y_true, y_score)
        stats["roc_auc"] = float(auc(fpr, tpr))

        fig, ax = plt.subplots(figsize=(7,5))
        ax.plot(fpr, tpr)
        ax.set_title(f"{prefix} ROC, AUC={stats['roc_auc']:.4f}")
        ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"{prefix}_roc.png"), dpi=150)
        plt.close(fig)

        prec, rec, _ = precision_recall_curve(y_true, y_score)
        stats["ap"] = float(average_precision_score(y_true, y_score))

        fig, ax = plt.subplots(figsize=(7,5))
        ax.plot(rec, prec)
        ax.set_title(f"{prefix} PR, AP={stats['ap']:.4f}")
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"{prefix}_pr.png"), dpi=150)
        plt.close(fig)

    else:
        y_bin = label_binarize(y_true, classes=np.arange(C))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), y_proba.ravel())
        stats["roc_auc_micro"] = float(auc(fpr, tpr))

        fig, ax = plt.subplots(figsize=(7,5))
        ax.plot(fpr, tpr)
        ax.set_title(f"{prefix} ROC micro, AUC={stats['roc_auc_micro']:.4f}")
        ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"{prefix}_roc_micro.png"), dpi=150)
        plt.close(fig)

        prec, rec, _ = precision_recall_curve(y_bin.ravel(), y_proba.ravel())
        stats["ap_micro"] = float(average_precision_score(y_bin, y_proba, average="micro"))

        fig, ax = plt.subplots(figsize=(7,5))
        ax.plot(rec, prec)
        ax.set_title(f"{prefix} PR micro, AP={stats['ap_micro']:.4f}")
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, f"{prefix}_pr_micro.png"), dpi=150)
        plt.close(fig)

        stats["roc_auc_macro_ovr"] = float(roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"))

    with open(os.path.join(out_dir, f"{prefix}_auc.json"), "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)

def train_single_task_vit(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    label_col: str,
    class_names: list[str],
    out_dir: str,
    backbone: str = "google/vit-base-patch16-224-in21k",
    epochs: int = 5,
    batch_size: int = 128,
    lr: float = 2e-5,
    wd: float = 0.01,
    workers: int = 2,
    seed: int = 42,
):
    os.makedirs(out_dir, exist_ok=True)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoImageProcessor.from_pretrained(backbone, use_fast=True)

    tr_ds = SingleTaskImageDataset(train_df, processor, label_col)
    te_ds = SingleTaskImageDataset(test_df, processor, label_col)
    if len(tr_ds) == 0 or len(te_ds) == 0:
        raise ValueError(f"Empty dataset: train={len(tr_ds)} test={len(te_ds)}")

    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  num_workers=workers, collate_fn=collate_single)
    te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False, num_workers=workers, collate_fn=collate_single)

    model = ViTClassifier(backbone, n_classes=len(class_names)).to(device)

    # class weights separate for train and test (as requested)
    w_tr = make_class_weights(train_df[label_col].to_numpy(), len(class_names)).to(device)
    w_te = make_class_weights(test_df[label_col].to_numpy(),  len(class_names)).to(device)

    with open(os.path.join(out_dir, "class_weights_train_test.json"), "w") as f:
        json.dump({"train": w_tr.detach().cpu().tolist(), "test": w_te.detach().cpu().tolist()}, f, indent=2)

    crit_tr = nn.CrossEntropyLoss(weight=w_tr)
    crit_te = nn.CrossEntropyLoss(weight=w_te)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    hist = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}
    best_val = float("inf")

    for ep in range(1, epochs+1):
        # train
        model.train()
        tl, correct, total = 0.0, 0, 0
        for x, y in tqdm(tr_loader, desc=f"Epoch {ep}/{epochs} [train]"):
            x = x.to(device); y = y.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = crit_tr(logits, y)
            loss.backward()
            opt.step()

            tl += float(loss.item())
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            total += y.numel()

        train_loss = tl / max(1, len(tr_loader))
        train_acc  = correct / max(1, total)

        # val
        model.eval()
        vl, vcorrect, vtotal = 0.0, 0, 0
        with torch.no_grad():
            for x, y in tqdm(te_loader, desc=f"Epoch {ep}/{epochs} [val]"):
                x = x.to(device); y = y.to(device)
                logits = model(x)
                loss = crit_te(logits, y)
                vl += float(loss.item())
                pred = logits.argmax(1)
                vcorrect += (pred == y).sum().item()
                vtotal += y.numel()

        val_loss = vl / max(1, len(te_loader))
        val_acc  = vcorrect / max(1, vtotal)

        hist["train_loss"].append(train_loss)
        hist["train_acc"].append(train_acc)
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)

        print(f"\nEpoch {ep}/{epochs} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} | train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), os.path.join(out_dir, "best.pt"))

    with open(os.path.join(out_dir, "history.json"), "w") as f:
        json.dump(hist, f, indent=2)

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7,5))
    ax.plot(hist["train_loss"], label="train_loss")
    ax.plot(hist["val_loss"], label="val_loss")
    ax.legend(); ax.set_title("Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, "loss.png"), dpi=150); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7,5))
    ax.plot(hist["train_acc"], label="train_acc")
    ax.plot(hist["val_acc"], label="val_acc")
    ax.legend(); ax.set_title("Accuracy"); ax.set_xlabel("Epoch"); ax.set_ylabel("Acc")
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, "acc.png"), dpi=150); plt.close(fig)

    # final eval train+test
    model.load_state_dict(torch.load(os.path.join(out_dir, "best.pt"), map_location=device))
    model.eval()

    tr_eval_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=False, num_workers=workers, collate_fn=collate_single)
    y_tr, p_tr = predict_proba(model, tr_eval_loader, device)
    y_te, p_te = predict_proba(model, te_loader, device)

    eval_dir = os.path.join(out_dir, "final_eval")
    save_curves_and_reports(y_tr, p_tr, class_names, eval_dir, "train")
    save_curves_and_reports(y_te, p_te, class_names, eval_dir, "test")

    print("Saved to:", out_dir)
    return model


In [ ]:
# === Train Emotion-only ViT ===
# Якщо буде worker error — постав workers=0
emotion_out = os.path.join(OUT_ROOT, "runs", "vit_emotion")
model_emo = train_single_task_vit(
    train_df=emo_train_df,
    test_df=emo_test_df,
    label_col="label",
    class_names=EMO_CLASSES_7,
    out_dir=emotion_out,
    epochs=10,
    batch_size=32,
    lr=2e-5,
    wd=0.01,
    workers=2,
)


In [ ]:
# === UTKFace: build Age-only + Gender-only DFs (80/20 split) ===
AGE_BINS_9 = [(0,2),(3,9),(10,19),(20,29),(30,39),(40,49),(50,59),(60,69),(70,120)]
age_names = [f"{a}-{b}" for a,b in AGE_BINS_9]

def age_to_bin(age: int) -> int:
    for i,(a0,a1) in enumerate(AGE_BINS_9):
        if a0 <= age <= a1:
            return i
    return len(AGE_BINS_9)-1

# UTK: 0=male, 1=female -> we want 0=woman,1=man
UTK_GENDER_TO_ID = {0: 1, 1: 0}
GENDER_NAMES = ["woman","man"]

def collect_utk(root: str) -> pd.DataFrame:
    rows=[]
    for dp,_,files in os.walk(root):
        for fn in files:
            if not fn.lower().endswith(IMG_EXT):
                continue
            base = fn.split(".")[0]
            parts = base.split("_")
            if len(parts) < 2:
                continue
            try:
                age = int(parts[0])
                g_raw = int(parts[1])
                if g_raw not in UTK_GENDER_TO_ID:
                    continue
                rows.append({
                    "image_path": os.path.join(dp, fn),
                    "age_bin": age_to_bin(age),
                    "gender": UTK_GENDER_TO_ID[g_raw],
                })
            except:
                continue
    return pd.DataFrame(rows)

utk_df = collect_utk(utk_root)
print("UTK total:", len(utk_df))

# remove broken
utk_df, bad_utk = filter_bad_decode(utk_df, desc="clean UTK (decode)")
print("Removed broken UTK:", len(bad_utk), "Remaining:", len(utk_df))

age_train, age_test = train_test_split(
    utk_df[["image_path","age_bin"]].rename(columns={"age_bin":"label"}),
    test_size=0.2, random_state=42, shuffle=True, stratify=utk_df["age_bin"]
)

gen_train, gen_test = train_test_split(
    utk_df[["image_path","gender"]].rename(columns={"gender":"label"}),
    test_size=0.2, random_state=42, shuffle=True, stratify=utk_df["gender"]
)

print("Age train/test:", len(age_train), len(age_test))
print("Gender train/test:", len(gen_train), len(gen_test))


In [ ]:
# === Train Age-only ViT ===
age_out = os.path.join(OUT_ROOT, "runs", "vit_age")
model_age = train_single_task_vit(
    train_df=age_train,
    test_df=age_test,
    label_col="label",
    class_names=age_names,
    out_dir=age_out,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    wd=0.01,
    workers=2,
)


In [ ]:
import os
from google.colab import files

# paths where best weights are saved
emotion_best = os.path.join(OUT_ROOT, "runs", "vit_emotion", "best.pt")
age_best     = os.path.join(OUT_ROOT, "runs", "vit_age", "best.pt")

print("Emotion:", emotion_best, "exists:", os.path.exists(emotion_best))
print("Age    :", age_best,     "exists:", os.path.exists(age_best))

# download to your PC
files.download(emotion_best)
files.download(age_best)


In [ ]:
# === Train Gender-only ViT ===
gender_out = os.path.join(OUT_ROOT, "runs", "vit_gender")
model_gender = train_single_task_vit(
    train_df=gen_train,
    test_df=gen_test,
    label_col="label",
    class_names=GENDER_NAMES,
    out_dir=gender_out,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    wd=0.01,
    workers=2,
)


In [ ]:
import os
from google.colab import files

# paths where best weights are saved
gender_best = os.path.join(OUT_ROOT, "runs", "vit_gender", "best.pt")

print("Gender:", gender_best, "exists:", os.path.exists(gender_best))

# download to your PC
files.download(gender_best)


In [ ]:
# === YOLO: filter dataset (remove Contempt & Sleepy) + train ===
# This block creates OUT_ROOT/yolo_filtered and prepares Ultralytics format.
# Training is commented out—uncomment when ready.

import yaml
from ultralytics import YOLO

EXCLUDE = {"contempt", "sleepy"}

def find_file(root, name):
    for dp, _, files in os.walk(root):
        if name in files:
            return os.path.join(dp, name)
    raise FileNotFoundError(f"{name} not found under {root}")

def read_lines(p):
    with open(p, "r", encoding="utf-8") as f:
        return [ln.strip() for ln in f if ln.strip()]

def write_lines(p, lines):
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

yaml_path = find_file(yolo_root, "dataset.yaml")
base_root = os.path.dirname(yaml_path)

with open(yaml_path, "r", encoding="utf-8") as f:
    ycfg = yaml.safe_load(f)

names = ycfg.get("names")
if isinstance(names, dict):
    names = [names[i] for i in sorted(names.keys())]
names_l = [str(n).strip().lower() for n in names]

drop_ids = {i for i,n in enumerate(names_l) if n in EXCLUDE}
keep = [(i,n) for i,n in enumerate(names_l) if i not in drop_ids]
old2new = {old_i: new_i for new_i,(old_i,_) in enumerate(keep)}
new_names = [n for _,n in keep]

print("Original names:", names_l)
print("Dropping ids:", drop_ids)
print("New names:", new_names)

train_txt = find_file(yolo_root, "train.txt")
val_txt   = find_file(yolo_root, "val.txt")
test_txt  = find_file(yolo_root, "test.txt")

def abs_img_path(list_file, item):
    if os.path.isabs(item):
        return item
    p1 = os.path.normpath(os.path.join(os.path.dirname(list_file), item))
    if os.path.exists(p1):
        return p1
    p2 = os.path.normpath(os.path.join(base_root, item))
    if os.path.exists(p2):
        return p2
    p3 = os.path.normpath(os.path.join(yolo_root, item))
    if os.path.exists(p3):
        return p3
    return p1

def image_to_label(img_abs: str):
    rel = os.path.relpath(img_abs, base_root)
    rel2 = rel.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)
    rel2 = os.path.splitext(rel2)[0] + ".txt"
    cand = os.path.join(base_root, rel2)
    if os.path.exists(cand):
        return cand
    cand2 = os.path.splitext(img_abs)[0] + ".txt"
    if os.path.exists(cand2):
        return cand2
    base = os.path.splitext(os.path.basename(img_abs))[0] + ".txt"
    for dp, _, files in os.walk(base_root):
        if os.path.basename(dp).lower() == "labels" and base in files:
            return os.path.join(dp, base)
    return cand

out_root = os.path.join(OUT_ROOT, "yolo_filtered")
if os.path.exists(out_root):
    shutil.rmtree(out_root)
os.makedirs(out_root, exist_ok=True)

def filter_split(list_path: str):
    kept = []
    for item in read_lines(list_path):
        img_abs = abs_img_path(list_path, item)
        if not os.path.exists(img_abs):
            continue
        lab_abs = image_to_label(img_abs)
        if not os.path.exists(lab_abs):
            continue

        new_lab_lines = []
        with open(lab_abs, "r", encoding="utf-8") as f:
            for ln in f:
                ln = ln.strip()
                if not ln:
                    continue
                parts = ln.split()
                cid = int(float(parts[0]))
                if cid in drop_ids:
                    continue
                parts[0] = str(old2new[cid])
                new_lab_lines.append(" ".join(parts))

        if len(new_lab_lines) == 0:
            continue

        rel_img = os.path.relpath(img_abs, base_root)
        if os.sep + "images" + os.sep in rel_img:
            rel_lab = rel_img.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)
            rel_lab = os.path.splitext(rel_lab)[0] + ".txt"
        else:
            rel_lab = os.path.join("labels", os.path.splitext(os.path.basename(rel_img))[0] + ".txt")

        kept.append((img_abs, rel_img, rel_lab, new_lab_lines))
    return kept

train_items = filter_split(train_txt)
val_items   = filter_split(val_txt)
test_items  = filter_split(test_txt)

print("Kept:", {"train": len(train_items), "val": len(val_items), "test": len(test_items)})

def materialize(items, split_name: str):
    out_list = []
    for img_abs, rel_img, rel_lab, lab_lines in items:
        dst_img = os.path.join(out_root, rel_img)
        dst_lab = os.path.join(out_root, rel_lab)
        os.makedirs(os.path.dirname(dst_img), exist_ok=True)
        os.makedirs(os.path.dirname(dst_lab), exist_ok=True)
        shutil.copy2(img_abs, dst_img)
        write_lines(dst_lab, lab_lines)
        out_list.append(rel_img)
    write_lines(os.path.join(out_root, f"{split_name}.txt"), out_list)

materialize(train_items, "train")
materialize(val_items, "val")
materialize(test_items, "test")

new_yaml = {
    "path": out_root,
    "train": "train.txt",
    "val": "val.txt",
    "test": "test.txt",
    "nc": len(new_names),
    "names": new_names,
}
with open(os.path.join(out_root, "dataset.yaml"), "w", encoding="utf-8") as f:
    yaml.safe_dump(new_yaml, f, sort_keys=False)

print("Filtered YOLO dataset:", out_root)

# ---- Train YOLO (uncomment to train) ----
# model = YOLO("yolov8n.pt")
# model.train(data=os.path.join(out_root, "dataset.yaml"), epochs=50, imgsz=640, batch=16, device=0)
